<a href="https://colab.research.google.com/github/graham-gc/infosys722/blob/main/Spark_DataFrame___Basics_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spark DataFrame Basics

This notebook introduces the fundamentals of the Spark DataFrame. You will learn how to start a Spark session, read data into a DataFrame, explore its contents and structure, and manipulate it using both the DataFrame API and SQL syntax.

## Step 1: Install Java and PySpark

In [23]:
# Install Java 17 (Spark 4 will not start on Java 11 or older).
# apt's output is hidden because Colab's extra repositories print harmless warnings;
# Step 2 checks that Java really was installed.
!apt-get update -qq > /dev/null 2>&1
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-17-jdk-headless > /dev/null 2>&1

# Colab preinstalls dataproc-spark-connect, which pins PySpark to 4.0.x. This notebook
# runs Spark locally and does not use it, so remove it to avoid a dependency-conflict error.
!pip uninstall -y -qq dataproc-spark-connect

# Install PySpark, pinned to the current release so results are reproducible.
!pip install -q "pyspark==4.2.0"

## Step 2: Tell Python where Java is, and check the versions


In [24]:
import glob
import os
import subprocess
import sys

# Look for an installed JDK that Spark 4 supports, preferring Java 17.
java_home = None
for version in ("17", "21", "25"):
    matches = sorted(glob.glob(f"/usr/lib/jvm/java-{version}-openjdk-*"))
    if matches:
        java_home = matches[0]
        break

# Otherwise fall back to a JAVA_HOME that was already set (e.g. on your own machine).
if java_home is None:
    java_home = os.environ.get("JAVA_HOME")
if java_home is None:
    raise RuntimeError(
        "No Java 17+ found. Re-run the installation cell. If it still fails, run "
        "'!apt-get install -y openjdk-17-jdk-headless' on its own to see apt's error, "
        "or set JAVA_HOME yourself on your own machine."
    )

os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = os.path.join(java_home, "bin") + os.pathsep + os.environ["PATH"]

import pyspark

print("Python   :", sys.version.split()[0])
print("PySpark  :", pyspark.__version__)
print("JAVA_HOME:", java_home)
# 'java -version' prints to stderr rather than stdout, so we capture stderr.
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

Python   : 3.13.15
PySpark  : 4.2.0
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment (build 17.0.20.1+1-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.20.1+1-1-24.04-Ubuntu, mixed mode, sharing)



## Step 3: Start a SparkSession



In [25]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("SparkDataFrameBasics")
    .master("local[*]")                            # run locally, one thread per CPU core
    .config("spark.sql.shuffle.partitions", "4")   # small data, so few shuffle partitions
    .getOrCreate()                                 # reuse the session if one already exists
)

spark.sparkContext.setLogLevel("ERROR")            # hide INFO and WARN log messages

print("Spark version:", spark.version)
spark

Spark version: 4.2.0


## Step 4: Make the dataset available



In [26]:
from pathlib import Path

# Set to False to skip Google Drive and use a local sample file instead.
USE_GOOGLE_DRIVE = True

# Edit this to match where people.json lives in your Google Drive.
DRIVE_PATH = Path("/content/drive/MyDrive/Infosys722/Datasets/people.json")

# Fallback location, used when the Drive copy is not available.
LOCAL_PATH = Path("Datasets/people.json")

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        print("Not running in Colab, so Google Drive is not mounted.")

if USE_GOOGLE_DRIVE and DRIVE_PATH.exists():
    DATA_PATH = str(DRIVE_PATH)
else:
    # Recreate the sample data in JSON Lines format: one JSON object per line.
    LOCAL_PATH.parent.mkdir(parents=True, exist_ok=True)
    LOCAL_PATH.write_text(
        '{"name":"Michael"}\n'
        '{"name":"Andy", "age":30}\n'
        '{"name":"Justin", "age":19}\n'
    )
    DATA_PATH = str(LOCAL_PATH.resolve())
    print("Drive copy not used, so a sample file was written locally.")

print("Using data file:", DATA_PATH)
print("\nRaw file contents:")
print(Path(DATA_PATH).read_text())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using data file: /content/drive/MyDrive/Infosys722/Datasets/people.json

Raw file contents:
{"name":"Michael"}
{"name":"Andy", "age":30}
{"name":"Justin", "age":19}



## Step 5: Read the data and let Spark infer the schema



In [27]:
# Read the JSON Lines file. Spark scans it to infer column names and types.
df = spark.read.json(DATA_PATH)

## Data Exploration

### Step 6: Look at the rows and the columns



In [28]:
# show() is an action: Spark reads the data and prints the first rows.
df.show()

# Column names come from the schema Spark already knows, so no job runs here.
print("Columns:", df.columns)

# count() is an action: Spark runs a job to count the rows.
print("Row count:", df.count())

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+

Columns: ['age', 'name']
Row count: 3


### Step 7: Summary statistics with describe() and summary()



In [29]:
# describe() returns a DataFrame of statistics, so show() is needed to display it.
df.describe().show()

# summary() adds the 25%, 50% (median) and 75% percentiles.
df.summary().show()

+-------+------------------+-------+
|summary|               age|   name|
+-------+------------------+-------+
|  count|                 2|      3|
|   mean|              24.5|   NULL|
| stddev|7.7781745930520225|   NULL|
|    min|                19|   Andy|
|    max|                30|Michael|
+-------+------------------+-------+

+-------+------------------+-------+
|summary|               age|   name|
+-------+------------------+-------+
|  count|                 2|      3|
|   mean|              24.5|   NULL|
| stddev|7.7781745930520225|   NULL|
|    min|                19|   Andy|
|    25%|                19|   NULL|
|    50%|                19|   NULL|
|    75%|                30|   NULL|
|    max|                30|Michael|
+-------+------------------+-------+



### Step 8: Check the data types with printSchema()



In [30]:
df.printSchema()

# The same information as a Python list of (column name, type) pairs.
print(df.dtypes)

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)

[('age', 'bigint'), ('name', 'string')]


## Data Manipulation

### Step 9: Define the schema yourself



In [31]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

# Each StructField is (column name, data type, nullable).
people_schema = StructType([
    StructField("age", IntegerType(), True),
    StructField("name", StringType(), True),
])

# Read the same file again, this time telling Spark exactly what to expect.
df = spark.read.schema(people_schema).json(DATA_PATH)

df.printSchema()
df.show()

root
 |-- age: integer (nullable = true)
 |-- name: string (nullable = true)

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



In [32]:
# The same schema as a DDL string, the syntax used in SQL CREATE TABLE.
ddl_schema = "age INT, name STRING"

df_ddl = spark.read.schema(ddl_schema).json(DATA_PATH)
df_ddl.printSchema()

# Both approaches produce an identical schema.
print("Schemas match:", df_ddl.schema == df.schema)

root
 |-- age: integer (nullable = true)
 |-- name: string (nullable = true)

Schemas match: True


In [33]:
# Read with the inferred schema (age is long), then cast age to a 32-bit integer.
df_cast = spark.read.json(DATA_PATH).withColumn("age", F.col("age").cast("int"))
df_cast.printSchema()

# try_cast returns NULL when a value cannot be converted, instead of raising an error.
spark.range(1).select(F.lit("thirty").try_cast("int").alias("age")).show()

# Uncomment to see ANSI mode reject a bad cast with a CAST_INVALID_INPUT error.
# spark.range(1).select(F.lit("thirty").cast("int").alias("age")).show()

root
 |-- age: integer (nullable = true)
 |-- name: string (nullable = true)

+----+
| age|
+----+
|NULL|
+----+



### Step 10: Select columns



In [34]:
# Transformation and action chained on one line.
df.select("age").show()

# The same in two steps: select() only builds a plan, show() runs it.
age_column = df.select("age")
age_column.show()

# Select several columns in a chosen order, plus a new column built from an expression.
df.select(
    F.col("name"),
    F.col("age"),
    (F.col("age") + 1).alias("age_next_year"),
).show()

+----+
| age|
+----+
|NULL|
|  30|
|  19|
+----+

+----+
| age|
+----+
|NULL|
|  30|
|  19|
+----+

+-------+----+-------------+
|   name| age|age_next_year|
+-------+----+-------------+
|Michael|NULL|         NULL|
|   Andy|  30|           31|
| Justin|  19|           20|
+-------+----+-------------+



### Step 11: Add columns with withColumn()



In [35]:
# withColumn returns a NEW DataFrame with the extra column.
df.withColumn("double_age", F.col("age") * 2).show()

# The original DataFrame is unchanged, because DataFrames are immutable.
df.show()

# To keep the result, assign it to a variable.
df_with_double = df.withColumn("double_age", F.col("age") * 2)
df_with_double.show()

# Add several columns in one step with withColumns and a dictionary.
df_enriched = df.withColumns({
    "double_age": F.col("age") * 2,
    "is_adult": F.col("age") >= 18,
})
df_enriched.show()

+----+-------+----------+
| age|   name|double_age|
+----+-------+----------+
|NULL|Michael|      NULL|
|  30|   Andy|        60|
|  19| Justin|        38|
+----+-------+----------+

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+

+----+-------+----------+
| age|   name|double_age|
+----+-------+----------+
|NULL|Michael|      NULL|
|  30|   Andy|        60|
|  19| Justin|        38|
+----+-------+----------+

+----+-------+----------+--------+
| age|   name|double_age|is_adult|
+----+-------+----------+--------+
|NULL|Michael|      NULL|    NULL|
|  30|   Andy|        60|    true|
|  19| Justin|        38|    true|
+----+-------+----------+--------+



### Step 12: Rename columns



In [36]:
df.withColumnRenamed("age", "my_new_age").show()

# Rename several columns at once.
df.withColumnsRenamed({"age": "age_years", "name": "full_name"}).show()

# A misspelled column name is silently ignored: no error, and no rename.
df.withColumnRenamed("agee", "my_new_age").show()

+----------+-------+
|my_new_age|   name|
+----------+-------+
|      NULL|Michael|
|        30|   Andy|
|        19| Justin|
+----------+-------+

+---------+---------+
|age_years|full_name|
+---------+---------+
|     NULL|  Michael|
|       30|     Andy|
|       19|   Justin|
+---------+---------+

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



## Introducing SQL

### Step 13: Register the DataFrame as a temporary view



In [37]:
# Give the DataFrame a name that SQL queries can refer to.
df.createOrReplaceTempView("people")

# spark.sql returns a DataFrame, so show() is still needed to see the rows.
results = spark.sql("SELECT * FROM people")
results.show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



### Step 14: Filter rows with WHERE, and pass values safely



In [38]:
spark.sql("SELECT age FROM people WHERE age >= 19").show()

# Rows with a missing age must be matched with IS NULL.
spark.sql("SELECT name FROM people WHERE age IS NULL").show()

# Parameterised query: the value is passed separately from the SQL text.
min_age = 19
spark.sql(
    "SELECT name, age FROM people WHERE age >= :min_age",
    args={"min_age": min_age},
).show()

+---+
|age|
+---+
| 30|
| 19|
+---+

+-------+
|   name|
+-------+
|Michael|
+-------+

+------+---+
|  name|age|
+------+---+
|  Andy| 30|
|Justin| 19|
+------+---+



### Step 15: The same query in DataFrame syntax



In [39]:
# DataFrame version of: SELECT name, age FROM people WHERE age >= 19
df_version = df.filter(F.col("age") >= 19).select("name", "age")
df_version.show()

sql_version = spark.sql("SELECT name, age FROM people WHERE age >= 19")

# Both styles compile to the same physical plan.
df_version.explain()
sql_version.explain()

+------+---+
|  name|age|
+------+---+
|  Andy| 30|
|Justin| 19|
+------+---+

== Physical Plan ==
*(1) Project [name#863, age#862]
+- *(1) Filter (isnotnull(age#862) AND (age#862 >= 19))
   +- FileScan json [age#862,name#863] Batched: false, DataFilters: [isnotnull(age#862), (age#862 >= 19)], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/content/drive/MyDrive/Infosys722/Datasets/people.json], PartitionFilters: [], PushedFilters: [IsNotNull(age), GreaterThanOrEqual(age,19)], ReadSchema: struct<age:int,name:string>


== Physical Plan ==
*(1) Project [name#863, age#862]
+- *(1) Filter (isnotnull(age#862) AND (age#862 >= 19))
   +- FileScan json [age#862,name#863] Batched: false, DataFilters: [isnotnull(age#862), (age#862 >= 19)], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/content/drive/MyDrive/Infosys722/Datasets/people.json], PartitionFilters: [], PushedFilters: [IsNotNull(age), GreaterThanOrEqual(age,19)], ReadSchema: struct<age:int,name:string>




## Step 16: Stop the SparkSession



In [40]:
spark.stop()